# 02 · Tall column, then iron the outer faces with the second arm
Nino's experiment in four parts that can each be run on their own:

1. **Column**: one arm prints a tall cylinder (bed does the motion, as `src/cell/runs/cylinder.py`).
2. **Face tracking, cold and clear**: the second arm follows the outer face at a set height, a few millimetres off it, while the bed turns. Proves the geometry before anything touches.
3. **Ironing pass**: the second arm's nozzle (hot, extruder off) presses the face by `PRESS` mm and sweeps a Z band while the bed turns.
4. **Opposite faces**: the arms sit 180° apart, so one face is ironed while the other is printed; then the bed brings the other round.

Cells marked **TODO** are the technique decisions: fill them in from the first trials.

Settings are in the first cell. Simulators first (`SIM = True`).

## Settings

In [ ]:
import sys, os, time, math, threading
ROOT = os.path.abspath("..")
sys.path[:0] = [ROOT, os.path.join(ROOT, "src"), os.path.join(ROOT, "ui")]   # not needed after: pip install -e .
import cell.runs.cylinder as cyl
from cell.runs.cylinder import Cell, PARK, SAFE_Z, SIDE, WATCHDOG_S

SIM = True
PRINT_ARM, IRON_ARM = 0, 1          # right prints, left irons (swap to test the other way)

COLUMN = dict(T=PRINT_ARM, R=40, E=4, L_H=0.6, HEIGHT=40, FLOW_K=1.3)     # cylinder_test settings
IRON = dict(
    CLEAR=3.0,       # face tracking: how far outside the face the nozzle tracks, mm
    PRESS=0.2,       # ironing: how far INSIDE the nominal face the nozzle sits, mm   TODO tune
    Z_FROM=5.0,      # ironing: band to iron, mm above Z0 of the ironing arm          TODO
    Z_TO=35.0,
    Z_RATE=0.3,      # ironing: mm of Z per bed revolution (a slow helix over the face)  TODO
    REVS=0,          # ironing: 0 = derive from the band and Z_RATE
    BED_DPS=20.0,    # ironing: bed speed while ironing, deg/s                          TODO
    TEMP=190,        # ironing: ironing nozzle temperature, C (soft but not dragging)   TODO
)

## Print the column (one arm)

In [ ]:
g, _ = cyl.parse([f"{k}={v}" for k, v in COLUMN.items()])
k = cyl.plan(g)
cyl.show(g, k)
# cell = Cell(g, SIM); cyl.run(g, k, cell)        # uncomment to print; same as: python3 cylinder_test.py T=0 R=40 E=4 ...

## Face tracking, cold and clear
The ironing arm goes to the column's radius plus `CLEAR`, at mid height, and holds there while the bed turns a revolution.
For a cylinder the point is fixed in the arm's frame (the face is a circle about the axis); for a rounded square it would be
`radius(a, θ) + CLEAR` updated every tick, exactly as in notebook 01. Watch the gap stay constant all the way round.

In [ ]:
from cell.runs.cylinder import DEFAULTS
g2 = {"arms": {IRON_ARM: dict(DEFAULTS[IRON_ARM], T=IRON_ARM, side=SIDE[IRON_ARM])}}
cell = Cell(g2, SIM)
ia = g2["arms"][IRON_ARM]
R_FACE = COLUMN["R"] + ia.get("DR", 0.0)               # the face this arm sees (its outline offset included)

def pt(a, r):
    ang = math.radians(a["ANGLE"]); return (a["CX"] + r * math.cos(ang), a["CY"] + r * math.sin(ang))

def iron_move(r, z, speed=10, wait=True):
    if z < ia["Z0"] - 1e-6: raise RuntimeError("below Z0")
    x, y = pt(ia, r); cell.move(IRON_ARM, x, y, z, speed=speed, wait=wait)

z_mid = ia["Z0"] + (IRON["Z_FROM"] + IRON["Z_TO"]) / 2
cell.move(IRON_ARM, *PARK, speed=60, wait=True)
cell.move(IRON_ARM, PARK[0], PARK[1], SAFE_Z, speed=60, wait=True)
x, y = pt(ia, R_FACE + 20); cell.move(IRON_ARM, x, y, SAFE_Z, speed=60, wait=True)
iron_move(R_FACE + 20, z_mid, speed=20)
iron_move(R_FACE + IRON["CLEAR"], z_mid, speed=5)       # creep in to the clearance
cell.tt.rotate_relative(360, 20, wait=True)             # one revolution: the gap should not change
iron_move(R_FACE + 20, z_mid, speed=10)
print("tracking pass done; measured gap constant? note it here:")

## Ironing pass
Nozzle hot, extruder **off**, pressed `PRESS` mm into the face, climbing `Z_RATE` mm per revolution from `Z_FROM` to `Z_TO` while the
bed turns at `BED_DPS`. Start with a small band and a small PRESS. **TODO**: decide hot vs warm, up vs down sweep, and whether
one pass or several.

In [ ]:
if not SIM:
    cell.ext.heat_and_wait(IRON_ARM, IRON["TEMP"])
cell.gcode("EXTRUDE_OFF")
band = IRON["Z_TO"] - IRON["Z_FROM"]
revs = IRON["REVS"] or band / IRON["Z_RATE"]
z0, z1 = ia["Z0"] + IRON["Z_FROM"], ia["Z0"] + IRON["Z_TO"]
print(f"ironing {band:.1f} mm over {revs:.1f} revs at {IRON['BED_DPS']:g} deg/s: {revs * 360 / IRON['BED_DPS']:.0f} s")

iron_move(R_FACE + IRON["CLEAR"], z0, speed=10)
iron_move(R_FACE - IRON["PRESS"], z0, speed=2)          # press in
a0 = cell.tt.get_angle()
cell.tt.rotate_velocity(IRON["BED_DPS"])
try:
    while True:
        th = cell.tt.get_angle() - a0
        if th >= revs * 360: break
        z = z0 + (z1 - z0) * th / (revs * 360)           # slow helix up the face
        iron_move(R_FACE - IRON["PRESS"], z, speed=5, wait=False)
        time.sleep(0.05)
finally:
    cell.tt.stop_rotation()
iron_move(R_FACE + IRON["CLEAR"], z1, speed=5)          # back off
iron_move(R_FACE + 20, z1, speed=10)
print("ironing pass done")

## Opposite faces while printing
The printing arm and the ironing arm are 180° apart, so while the column is being laid on one side the other side is a few layers
old and can be ironed. The questions to settle in trials: how many layers behind the print head the ironing band should sit
(`LAG_LAYERS`), and the back-off rule when the printing arm has to cross. Skeleton below; it reuses the ironing pass for the ironing arm and
`cylinder_test.run` for the printing arm in a thread. **Do not run until the column, the face tracking and the ironing pass are each proven on their own.**

In [ ]:
LAG_LAYERS = 10        # TODO: layers between the print head and the ironing band
# plan:
#   1. start cyl.run(g, k, cell_print) in a thread (printing arm, bed master as usual)
#   2. in this cell, follow z_print - LAG_LAYERS * L_H with the ironing arm at R_FACE - PRESS, extruder off, hot
#   3. the bed speed is set by the print; the ironing arm just follows z, so no second bed command
#   4. stop ironing when the print's final flat loop starts; back off; let the print finish
print("skeleton only")

## What to measure
- Surface: photo under raking light, ironed vs unironed faces; profile gauge if available.
- Wall thickness at the ironed and unironed faces (calipers), to see how much material the press displaces.
- Any drag marks or tearing → TEMP too high or PRESS too deep.
- Note PRESS, TEMP, BED_DPS, Z_RATE for every trial in the table below.

| trial | PRESS | TEMP | BED_DPS | Z_RATE | result |
|---|---|---|---|---|---|
| 1 | | | | | |